In [1]:
import torch
from scripts.NNModel import NeuralNetwork
import scripts.pyTorchTraining as pyTrain
from os.path import join
import yaml

Welcome to JupyROOT 6.28/00


In [2]:
# Inputs to be read from outside

tagger = "OSKaon"
cut_name = "notSamePV_noOSP"
features_file = "union_PROBNN"
repo = "/work/celani/flavour_tagging/classical-taggers"
seed = 45
decay_type = "Bu2JpsiK"
config_file = "lr0.001_bs128_simple_dm0.0"
model_savepath = "/work/celani/flavour_tagging/Micol_trainings/savedModels/withUT_MC_2024"

target_path = join(model_savepath, f"{decay_type}/{tagger}/{cut_name}/{features_file}/{seed}/{config_file}/model.pth")

In [3]:
# Create the a NN with the charachteristic of the saved model
features = pyTrain.get_features(tagger=tagger, yaml_file=features_file, repo_path=repo)
with open(join(repo, f'configs/{config_file}.yaml'), 'r') as file:
    config = yaml.safe_load(file)
model = NeuralNetwork(features=features, architecture=config['architecture'], seed=seed, optimizer_kwargs={"lr" : config['learning_rate']}, repo_path=repo).to("cpu")

In [5]:
model.load_state_dict(torch.load(f"{target_path}"))
model.eval()  # switch to inference mode

/tmp/ipykernel_440792/479348664.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"{target_path}"))


NeuralNetwork(
  (criterion): BCELoss()
  (NN): Sequential(
    (0): Linear(in_features=17, out_features=3, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=3, out_features=3, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=3, out_features=1, bias=True)
    (5): Sigmoid()
  )
)

Remember to 
pip install onnx
pip install onnxruntime

In [9]:
# Create a dummy input matching the input shape (len(features))
dummy_input = torch.randn(1, len(features))

# Export to ONNX
torch.onnx.export(
    model.NN,                            # Export the inner nn.Sequential
    dummy_input,                         # Dummy input
    "model.onnx",                        # Output file name
    input_names=["float_input"],         # Input name
    output_names=["output_probability"], # Output name
    dynamic_axes={
        "float_input": {0: "batch_size"},
        "output_probability": {0: "batch_size"},
    },
    # opset_version=11                     # Choose your ONNX opset version
)

In [10]:
# Check
import onnx

# Load the ONNX model
model_onnx = onnx.load("model.onnx")

# Inspect input and output
print("Inputs:")
for input in model_onnx.graph.input:
    print(f"  name: {input.name}")
    print(f"  type: {input.type}")

print("\nOutputs:")
for output in model_onnx.graph.output:
    print(f"  name: {output.name}")
    print(f"  type: {output.type}")

Inputs:
  name: float_input
  type: tensor_type {
  elem_type: 1
  shape {
    dim {
      dim_param: "batch_size"
    }
    dim {
      dim_value: 17
    }
  }
}


Outputs:
  name: output_probability
  type: tensor_type {
  elem_type: 1
  shape {
    dim {
      dim_param: "batch_size"
    }
    dim {
      dim_value: 1
    }
  }
}

